# Tutorial 03 — Hyperelastic Constitutive Models

This notebook compares sixteen educational constitutive models with synthetic parameters.

## Before running the calculations

The notebook imports the repository's own `biomechanics_tutorials` package from `src/`. It calculates every curve in memory; saved PNG/GIF files are created separately by `reproduce.py`.

In [ ]:
LANGUAGE = "en"

import importlib.util
import os
import sys
from pathlib import Path


def _is_repository_root(path: Path) -> bool:
    return (path / "pyproject.toml").is_file() and (
        path / "src" / "biomechanics_tutorials"
    ).is_dir()


def _find_repository_root() -> Path:
    candidates: list[Path] = []
    installed_spec = importlib.util.find_spec("biomechanics_tutorials")
    if installed_spec is not None and installed_spec.origin:
        package_file = Path(installed_spec.origin).resolve()
        if len(package_file.parents) >= 3:
            candidates.append(package_file.parents[2])
    environment_root = os.environ.get("BMRT_ROOT")
    if environment_root:
        candidates.append(Path(environment_root).expanduser())
    current = Path.cwd().resolve()
    candidates.extend([current, *current.parents])
    home = Path.home()
    common_directories = [home, home / "Desktop", home / "Documents", home / "Downloads"]
    for directory in common_directories:
        candidates.append(directory / "Biomechanics-Research-Tutorials")
        if directory.is_dir():
            candidates.extend(directory.glob("Biomechanics-Research-Tutorials*"))
    for candidate in candidates:
        candidate = candidate.resolve()
        if _is_repository_root(candidate):
            return candidate
    raise RuntimeError(
        "Repository root was not found. Extract the complete repository, start Jupyter "
        "from its root, or install it in the active environment with: "
        "python -m pip install -e .[dev]"
    )


REPOSITORY_ROOT = _find_repository_root()
SOURCE_DIRECTORY = REPOSITORY_ROOT / "src"
if str(SOURCE_DIRECTORY) not in sys.path:
    sys.path.insert(0, str(SOURCE_DIRECTORY))

import numpy as np
import matplotlib.pyplot as plt
import biomechanics_tutorials
from biomechanics_tutorials.hyperelasticity import (
    ISOTROPIC_MODELS,
    MODEL_DEFINITIONS,
    MYOCARDIUM_MODELS,
    deformation_gradient,
    kinematic_state,
    model_catalog,
    neo_hookean_uniaxial_nominal_stress,
    path_response,
    strain_energy,
    volumetric_energy,
)
from biomechanics_tutorials.plotting import apply_tutorial_style

apply_tutorial_style()
print("Repository root:", REPOSITORY_ROOT)
print("Python kernel:", sys.executable)
print("Local package:", Path(biomechanics_tutorials.__file__).resolve())

## Model catalog

In [ ]:
LABELS = {
    "stretch": {"en": "Stretch λ", "ru": "Растяжение λ"},
    "response": {"en": "Generalized response dW/dq", "ru": "Обобщённый отклик dW/dq"},
    "angle": {"en": "Fiber angle, degrees", "ru": "Угол волокон, градусы"},
    "volume": {"en": "Volume ratio J", "ru": "Отношение объёмов J"},
    "energy": {"en": "Energy", "ru": "Энергия"},
}

def label(key):
    return LABELS[key][LANGUAGE]

print("Number of implemented models:", len(model_catalog()))
for definition in model_catalog():
    print(f"{definition.key:24s} | {definition.category:11s} | {definition.name}")

## Kinematics and invariants

In [ ]:
F = deformation_gradient("uniaxial", 1.20)
state = kinematic_state(F)
print("F =\n", F)
print("J =", state["J"])
print("I1_bar =", state["I1_bar"])
print("I2_bar =", state["I2_bar"])
print("principal stretches =", state["principal_stretches_bar"])

## Ten isotropic models

In [ ]:
stretches = np.linspace(0.80, 1.50, 220)
plt.figure(figsize=(10, 6))
for key in ISOTROPIC_MODELS:
    plt.plot(stretches, path_response(key, stretches), linewidth=1.8,
             label=MODEL_DEFINITIONS[key].name)
plt.axhline(0, linewidth=0.8)
plt.axvline(1, linewidth=0.8)
plt.xlabel(label("stretch"))
plt.ylabel(label("response"))
plt.legend(ncol=2, fontsize=8)
plt.show()

## Multiple loading modes

In [ ]:
selected = ["neo_hookean", "mooney_rivlin", "gent", "ogden_2"]
modes = [
    ("uniaxial", np.linspace(1.0, 1.45, 150)),
    ("equibiaxial", np.linspace(1.0, 1.28, 150)),
    ("plane_strain", np.linspace(1.0, 1.45, 150)),
    ("simple_shear_xy", np.linspace(0.0, 0.8, 150)),
]
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, (mode, q) in zip(axes.flat, modes):
    for key in selected:
        ax.plot(q, path_response(key, q, mode), label=MODEL_DEFINITIONS[key].name)
    ax.set_title(mode)
    ax.set_ylabel(label("response"))
axes[0, 0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Fiber angle

In [ ]:
stretches = np.linspace(1.0, 1.35, 160)
plt.figure(figsize=(9, 5.5))
for angle in [0, 20, 40, 60, 80]:
    response = path_response("hgo", stretches, parameters={"fiber_angle_deg": angle})
    plt.plot(stretches, response, label=f"α={angle}°")
plt.xlabel(label("stretch"))
plt.ylabel(label("response"))
plt.legend()
plt.show()

## GOH dispersion

In [ ]:
plt.figure(figsize=(9, 5.5))
for kappa in [0.0, 0.08, 0.18, 1.0 / 3.0]:
    response = path_response("goh", stretches, parameters={"kappa": kappa})
    plt.plot(stretches, response, label=f"κ={kappa:.3f}")
plt.xlabel(label("stretch"))
plt.ylabel(label("response"))
plt.legend()
plt.show()

## Myocardial shear planes

In [ ]:
shear = np.linspace(0.0, 0.55, 140)
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
for ax, mode in zip(axes, ["simple_shear_xy", "simple_shear_xz", "simple_shear_yz"]):
    for key in MYOCARDIUM_MODELS:
        ax.plot(shear, path_response(key, shear, mode), label=MODEL_DEFINITIONS[key].name)
    ax.set_title(mode)
    ax.set_ylabel(label("response"))
axes[0].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Volumetric penalties

In [ ]:
J = np.linspace(0.75, 1.30, 200)
plt.figure(figsize=(9, 5.5))
for kind in ["quadratic", "logarithmic", "simo_taylor"]:
    plt.plot(J, volumetric_energy(J, 10.0, kind), label=kind)
plt.xlabel(label("volume"))
plt.ylabel(label("energy"))
plt.legend()
plt.show()

## Verification of the energy derivative

In [ ]:
stretches = np.linspace(0.80, 1.50, 180)
analytical = neo_hookean_uniaxial_nominal_stress(stretches)
numerical = path_response("neo_hookean", stretches)
print("maximum derivative error =", np.max(np.abs(analytical - numerical)))
plt.figure(figsize=(9, 5.5))
plt.plot(stretches, analytical, label="analytical")
plt.plot(stretches, numerical, "--", label="finite difference")
plt.xlabel(label("stretch"))
plt.ylabel(label("response"))
plt.legend()
plt.show()

## Reference-state check

In [ ]:
identity = np.eye(3)
energies_at_identity = {definition.key: strain_energy(definition.key, identity)
                        for definition in model_catalog()}
assert all(abs(value) < 1e-12 for value in energies_at_identity.values())
print("All sixteen energies are zero in the reference configuration.")

## Interpretation

A model that fits one loading mode may fail in another. These curves verify and compare mathematical behaviour; they do not validate tissue-specific parameters.